# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass ***Fine-tuning y Evaluación de Modelos***, se proporciona el siguiente ***Colab*** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo Llama ligero, y medir su mejora con una métrica objetiva.

A diferencia de los Temas anteriores, aquí no usamos Groq — Groq solo sirve para inferencia, no para entrenar modelos. Usamos **Hugging Face** (librerías `transformers` y `peft`) directamente sobre la GPU gratuita de Colab.

## **CONFIGURACIÓN DEL ENTORNO**

### **COLAB SECRETS**

Para no exponer tu ***token*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarlo de forma segura, añadiendo un nombre asociado al token para guardarlo dentro de una variable y usarlo dentro del notebook. Para este Tema necesitas un ***token de Hugging Face*** (el modelo que usamos es de acceso libre, no requiere solicitar permiso especial).

In [ ]:
# NO EJECUTAR PARA JUPITER EN LOCAL SOLO LA SIGUIENTE CELDA
# Instalar librerias e iniciar sesión en Hugging Face con el token desde Colab Secrets

!pip install transformers peft accelerate trl --quiet

import torch
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))
print("Sesión de Hugging Face iniciada correctamente.")

In [1]:
pip install transformers peft accelerate trl huggingface_hub

Note: you may need to restart the kernel to use updated packages.


In [19]:
# Para ejecutarlo en Jupyter local

import torch
import os
from huggingface_hub import login


hf_token = os.getenv("HF_TOKEN")  # lee la variable de entorno
login(token=hf_token)
print("Sesión de Hugging Face iniciada correctamente.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Sesión de Hugging Face iniciada correctamente.


### **CARGAR EL MODELO BASE**

Usamos un modelo Llama ligero (pocos parámetros) para que el fine-tuning corra en minutos sobre la GPU T4 gratuita de Colab, sin necesitar cuantización adicional.

In [20]:
# Cargar el modelo base de Llama y su tokenizer

from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# variante oficial de Meta "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Modelo base cargado: TinyLlama/TinyLlama-1.1B-Chat-v1.0


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [21]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo

def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split("\n")[0].strip()

prompt_prueba = "Estudiante: ¿Puedo entregar la tarea un día después?\nProfesor:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)
print(respuesta_base)

No, no puedes entregarlo un día antes. Debes entregármelo el día que estés listo.


### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que muestren el comportamiento que queremos enseñarle al modelo. Con pocos ejemplos (5 a 10) es suficiente para una demo — no es un dataset de producción.

In [33]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset

from datasets import Dataset
from transformers import AutoTokenizer
from trl import SFTTrainer, SFTConfig

# 1. Dataset académico
ejemplos = [
    {"texto": "Estudiante: ¿Puedo entregar la tarea un día después?\nProfesor: Sí, pero se aplicará una penalización del 10% en la calificación."},
    {"texto": "Estudiante: ¿Cuándo estarán disponibles las calificaciones del examen?\nProfesor: Se publicarán en la plataforma académica dentro de cinco días hábiles."},
    {"texto": "Estudiante: ¿Habrá clase el próximo viernes?\nProfesor: No, ese día está programado como jornada institucional sin actividades académicas."},
    {"texto": "Estudiante: ¿Puedo solicitar revisión de mi examen?\nProfesor: Sí, debes enviar una solicitud formal al correo del departamento en un plazo de 48 horas."},
    {"texto": "Estudiante: ¿Cómo puedo acceder a los materiales de la clase?\nProfesor: Están disponibles en la sección 'Recursos' de la plataforma institucional."},
    {"texto": "Estudiante: ¿Puedo cambiar de grupo en este semestre?\nCoordinador: Sí, siempre que haya cupo disponible y presentes la solicitud antes de la segunda semana."},
    {"texto": "Estudiante: ¿Qué hago si no puedo asistir a la práctica de laboratorio?\nProfesor: Debes justificar tu ausencia y solicitar una reposición en el siguiente turno disponible."},
    {"texto": "Estudiante: ¿Cómo obtengo constancia de estudios?\nCoordinador: Puedes solicitarla en el área de servicios escolares con tu número de matrícula."},
    {"texto": "Estudiante: ¿El proyecto final es individual o en equipo?\nProfesor: Es en equipo, con un máximo de cuatro integrantes por grupo."},
    {"texto": "Estudiante: ¿Puedo presentar el examen extraordinario?\nCoordinador: Sí, siempre que cumplas con el pago correspondiente y te registres en la fecha indicada."},
]

dataset = Dataset.from_list(ejemplos)

# 2. Tokenizador (ajusta al modelo que uses, ej. TinyLlama)
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token

# 3. Tokenización del dataset
def tokenize_function(example):
    return tokenizer(example["texto"], truncation=True, padding="max_length", max_length=128)

dataset_tokenizado = dataset.map(tokenize_function, batched=True)

# 4. División en train/test
dataset_split = dataset_tokenizado.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# 5. Configuración del entrenamiento
config_entrenamiento = SFTConfig(
    output_dir="./resultados",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
    save_strategy="epoch",
    use_cpu=True   # fuerza CPU
)

# 6. Entrenador
trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=config_entrenamiento,
)

# 7. Entrenamiento
resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

{'loss': '6.828', 'grad_norm': '45.49', 'learning_rate': '0.0002', 'entropy': '1.059', 'num_tokens': '256', 'mean_token_accuracy': '0.3228', 'epoch': '0.25'}
{'loss': '4.95', 'grad_norm': '55.59', 'learning_rate': '0.00019', 'entropy': '2.006', 'num_tokens': '512', 'mean_token_accuracy': '0.3071', 'epoch': '0.5'}
{'loss': '3.651', 'grad_norm': '42.29', 'learning_rate': '0.00018', 'entropy': '2.227', 'num_tokens': '768', 'mean_token_accuracy': '0.4213', 'epoch': '0.75'}
{'loss': '2.29', 'grad_norm': '34.14', 'learning_rate': '0.00017', 'entropy': '2.303', 'num_tokens': '1024', 'mean_token_accuracy': '0.5709', 'epoch': '1'}
{'loss': '1.381', 'grad_norm': '27.76', 'learning_rate': '0.00016', 'entropy': '2.167', 'num_tokens': '1280', 'mean_token_accuracy': '0.7362', 'epoch': '1.25'}
{'loss': '1.01', 'grad_norm': '18.09', 'learning_rate': '0.00015', 'entropy': '2.062', 'num_tokens': '1536', 'mean_token_accuracy': '0.8583', 'epoch': '1.5'}
{'loss': '0.5107', 'grad_norm': '11.56', 'learning_r

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [38]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base

!pip uninstall -y torchao --quiet

# 1. Cargar el modelo base desde Hugging Face
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from transformers import set_seed

set_seed(42)

modelo = AutoModelForCausalLM.from_pretrained(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
)
tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
tokenizer.pad_token = tokenizer.eos_token

# 2. Configurar LoRA
config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # módulos de atención
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

# 3. Aplicar LoRA al modelo base
modelo_lora = get_peft_model(modelo, config_lora)

# 4. Verificar parámetros entrenables
modelo_lora.print_trainable_parameters()
print(modelo_lora.peft_config)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

trainable params: 1,126,400 || all params: 1,101,174,784 || trainable%: 0.1023
{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path='TinyLlama/TinyLlama-1.1B-Chat-v1.0', revision=None, inference_mode=False, r=8, target_modules={'v_proj', 'q_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bd

### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [29]:
# Configurar el entrenador (SFTTrainer) y ejecutar el fine-tuning

from trl import SFTTrainer, SFTConfig

#config_entrenamiento = SFTConfig(
#    output_dir="/content/resultados",
#    num_train_epochs=30,
#    per_device_train_batch_size=5,
#    learning_rate=2e-4,
#    logging_steps=1,
#    dataset_text_field="texto",
#    max_length=128,
#    report_to="none",
#)

config_entrenamiento = SFTConfig(
    output_dir="./resultados",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=128,
    report_to="none",
    save_strategy="epoch",
    use_cpu=True   # fuerza CPU
)


trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=dataset,
    args=config_entrenamiento,
)

resultado_entrenamiento = trainer.train()
print("Pérdida final:", resultado_entrenamiento.training_loss)

Adding EOS to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

{'loss': '0.7522', 'grad_norm': '3.49', 'learning_rate': '0.0002', 'entropy': '1.092', 'num_tokens': '96', 'mean_token_accuracy': '0.8298', 'epoch': '0.2'}
{'loss': '0.7727', 'grad_norm': '5.056', 'learning_rate': '0.000192', 'entropy': '1.11', 'num_tokens': '180', 'mean_token_accuracy': '0.8049', 'epoch': '0.4'}
{'loss': '0.6862', 'grad_norm': '3.989', 'learning_rate': '0.000184', 'entropy': '1.073', 'num_tokens': '276', 'mean_token_accuracy': '0.8298', 'epoch': '0.6'}
{'loss': '0.6328', 'grad_norm': '4.13', 'learning_rate': '0.000176', 'entropy': '1.113', 'num_tokens': '370', 'mean_token_accuracy': '0.8261', 'epoch': '0.8'}
{'loss': '0.7952', 'grad_norm': '4.554', 'learning_rate': '0.000168', 'entropy': '1.131', 'num_tokens': '464', 'mean_token_accuracy': '0.7826', 'epoch': '1'}
{'loss': '0.5295', 'grad_norm': '3.817', 'learning_rate': '0.00016', 'entropy': '0.9689', 'num_tokens': '550', 'mean_token_accuracy': '0.8929', 'epoch': '1.2'}
{'loss': '0.5197', 'grad_norm': '3.725', 'learni

In [39]:
inputs = tokenizer(prompt_prueba, return_tensors="pt")

outputs = modelo_lora.generate(
    **inputs,
    max_new_tokens=80,        # límite de tokens nuevos
    temperature=0.7,          # controla aleatoriedad (0.7 es equilibrado)
    top_p=0.9,                # muestreo por probabilidad acumulada
    top_k=50,                 # limita a los 50 tokens más probables
    repetition_penalty=1.2,   # penaliza repetir tokens
    do_sample=True            # activa muestreo
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Estudiante: ¿Puedo entregar la tarea un día después?
Profesor: No, debí dejarte esperarlo. Deberías haber estado trabajando en tu trabajo.


### **DESPUÉS DEL FINE-TUNING: MEDIR LA MEJORA**

Compararemos la pérdida antes y después del entrenamiento como métrica objetiva.



In [40]:
# Comparar la pérdidas

perdida_inicial = trainer.state.log_history[0]['loss']
perdida_final = resultado_entrenamiento.training_loss

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")
print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")
print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

# trainer.state.log_history[0]['loss'] es la pérdida después del primer paso registrado,
# no la pérdida real del modelo sin ningún entrenamiento.

Pérdida al inicio del entrenamiento: 6.83
Pérdida final del entrenamiento: 1.14
Reducción: 83%


In [41]:
# Referencia cualitativa vs Referencia con un modelo de producción

!pip install groq -q
from groq import Groq

respuesta_ajustada = generar_respuesta(modelo_lora, prompt_prueba)
print("Respuesta del modelo ajustado (TinyLlama + LoRA):\n", respuesta_ajustada)

# Para ejecutar en Local con Jupyter 
api_key = os.getenv("GROQ_API_KEY")
client = Groq(api_key=api_key)

# Si se ejecuta en Colab 
#client = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_prueba + " Respuesta muy breve y corta."}]
)

print("\nRespuesta de referencia (Groq / GPT-OSS-20B):\n", response.choices[0].message.content)

Respuesta del modelo ajustado (TinyLlama + LoRA):
 No, no puedes entregarlo un día antes. Debes entregármelo el día que estés listo.

Respuesta de referencia (Groq / GPT-OSS-20B):
 Sí, avisa con antelación y justifica el retraso.


**Nota:** Por eso, además de la respuesta de TinyLlama, se muestra una respuesta de Groq (GPT-OSS-20B) para el mismo prompt: no porque el proceso de fine-tuning haya fallado, sino como punto de comparación de cómo respondería un modelo de producción con muchos más parámetros y datos de entrenamiento — TinyLlama con 5 ejemplos demuestra la técnica de LoRA, no busca igualar la calidad de un modelo así de grande.